In [ ]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "colab": {
      "provenance": []
    },
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "cells": [
    {
      "cell_type": "code",
      "execution_count": 1,
      "metadata": {
        "id": "JXFWkEzWKIg_"
      },
      "outputs": [],
      "source": [
        "!pip install pyspark -q"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "import os\n",
        "import shutil\n",
        "\n",
        "# Create Data Lake folders\n",
        "os.makedirs(\"data_lake/bronze\", exist_ok=True)\n",
        "os.makedirs(\"data_lake/silver/preprocessed_sales\", exist_ok=True)\n",
        "\n",
        "# Copy the original dataset into Bronze\n",
        "shutil.copy(\"sales.csv\", \"data_lake/bronze/sales.csv\")\n",
        "\n",
        "print(\"Data Lake structure created successfully!\")"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "ZCHkAHGpKspa",
        "outputId": "2b13fd77-677d-4c43-9baa-5535a1135568"
      },
      "execution_count": 2,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Data Lake structure created successfully!\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "from pyspark.sql import SparkSession\n",
        "\n",
        "spark = SparkSession.builder \\\n",
        "    .appName(\"Data Mining Data Lake Assignment\") \\\n",
        "    .getOrCreate()\n",
        "\n",
        "print(\"Spark version:\", spark.version)"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "PQsh96NdKzlq",
        "outputId": "742efc3a-1db3-4d07-c93c-8025de731010"
      },
      "execution_count": 3,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Spark version: 4.0.4\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "df = spark.read \\\n",
        "    .option(\"header\", True) \\\n",
        "    .option(\"inferSchema\", True) \\\n",
        "    .csv(\"data_lake/bronze/sales.csv\")\n",
        "\n",
        "print(\"Dataset loaded successfully!\")"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "bim4Ol2mK4E2",
        "outputId": "e41742b4-9195-459e-ad79-5cb7c83d7318"
      },
      "execution_count": 4,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Dataset loaded successfully!\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "df.printSchema()"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "Jgm9s1IQK_GZ",
        "outputId": "28f34709-e36a-4fd5-ab3c-4188959fd72f"
      },
      "execution_count": 5,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "root\n",
            " |-- order_id: integer (nullable = true)\n",
            " |-- customer_id: string (nullable = true)\n",
            " |-- customer_name: string (nullable = true)\n",
            " |-- product: string (nullable = true)\n",
            " |-- category: string (nullable = true)\n",
            " |-- quantity: integer (nullable = true)\n",
            " |-- unit_price: integer (nullable = true)\n",
            " |-- discount_percent: integer (nullable = true)\n",
            " |-- payment_method: string (nullable = true)\n",
            " |-- city: string (nullable = true)\n",
            " |-- state: string (nullable = true)\n",
            " |-- order_date: string (nullable = true)\n",
            " |-- order_status: string (nullable = true)\n",
            "\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "df.show(10, truncate=False)"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "UICH1UPgLJGu",
        "outputId": "8eca8551-6f3d-424e-d714-5c7780aea50e"
      },
      "execution_count": 6,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+\n",
            "|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|\n",
            "+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+\n",
            "|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |\n",
            "|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |\n",
            "|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |Pune      |Maharashtra   |2025-05-06|Shipped     |\n",
            "|100004  |C0051      |Tanvi Sharma   |Running Shoes        |Sports   |6       |2260      |35              |Net Banking     |Lucknow   |Uttar Pradesh |2026-07-31|Cancelled   |\n",
            "|100005  |C0227      |Aarav Kumar    |Coffee 500g          |Grocery  |10      |740       |12              |Debit Card      |Vijayawada|Andhra Pradesh|2026-08-30|Delivered   |\n",
            "|100006  |C0117      |Aditya Malhotra|Biography            |Books    |7       |650       |10              |Credit Card     |Pune      |Maharashtra   |2026-05-09|Shipped     |\n",
            "|100007  |C0187      |Siddharth Menon|Biography            |Books    |4       |730       |30              |Cash on Delivery|Hyderabad |Telangana     |2025-12-02|Delivered   |\n",
            "|100008  |C0124      |Tanvi Chaudhary|Dining Table         |Furniture|6       |25970     |40              |Cash on Delivery|Delhi     |Delhi         |2025-09-25|Delivered   |\n",
            "|100009  |C0320      |Lakshmi Rao    |Data Science Handbook|Books    |2       |680       |30              |Debit Card      |Vijayawada|Andhra Pradesh|2025-09-04|Delivered   |\n",
            "|100010  |C0113      |Neha Sharma    |Kurta                |Clothing |4       |860       |25              |UPI             |Lucknow   |Uttar Pradesh |2026-08-24|Processing  |\n",
            "+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+\n",
            "only showing top 10 rows\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "print(\"Total record count:\", df.count())"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "2-xbyH4FLQCM",
        "outputId": "ca852e4f-3032-4495-ba6b-cd6798c15f6e"
      },
      "execution_count": 7,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Total record count: 1000\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "from pyspark.sql.functions import col, sum\n",
        "\n",
        "null_counts = df.select([\n",
        "    sum(col(c).isNull().cast(\"int\")).alias(c)\n",
        "    for c in df.columns\n",
        "])\n",
        "\n",
        "null_counts.show()"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "kBBeHK-QLR72",
        "outputId": "1d56cf4c-bda7-4398-ae59-ff44ec4533f0"
      },
      "execution_count": 8,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "total_rows = df.count()\n",
        "distinct_rows = df.distinct().count()\n",
        "\n",
        "print(\"Total rows:\", total_rows)\n",
        "print(\"Distinct rows:\", distinct_rows)\n",
        "print(\"Exact duplicate rows:\", total_rows - distinct_rows)"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "A9phdTStLUu-",
        "outputId": "921cf941-60e3-4c29-900b-64cacbb5ed58"
      },
      "execution_count": 9,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Total rows: 1000\n",
            "Distinct rows: 990\n",
            "Exact duplicate rows: 10\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "print(\"Invalid quantity:\")\n",
        "df.filter(col(\"quantity\") <= 0).show()\n",
        "\n",
        "print(\"Invalid unit price:\")\n",
        "df.filter(col(\"unit_price\") < 0).show()\n",
        "\n",
        "print(\"Invalid discount percentage:\")\n",
        "df.filter(\n",
        "    (col(\"discount_percent\") < 0) |\n",
        "    (col(\"discount_percent\") > 100)\n",
        ").show()"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "3CKv66x3LXAS",
        "outputId": "402af15e-c721-4313-9dc7-52f8f6b4d42f"
      },
      "execution_count": 10,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Invalid quantity:\n",
            "+--------+-----------+-------------+--------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+\n",
            "|order_id|customer_id|customer_name|             product|       category|quantity|unit_price|discount_percent|  payment_method|      city|         state|order_date|order_status|\n",
            "+--------+-----------+-------------+--------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+\n",
            "|  100114|      C0252|Keerthi Singh|         Ceiling Fan|Home Appliances|       0|      2920|              15|     Net Banking|   Chennai|    Tamil Nadu|2025-02-15|   Delivered|\n",
            "|  100143|      C0184|   Aman Joshi|            Football|         Sports|       0|      1030|              15|      Debit Card|Coimbatore|    Tamil Nadu|2025-11-26|   Delivered|\n",
            "|  100198|      C0014| Arjun Pillai|              Hoodie|       Clothing|      -2|      2160|              12|     Net Banking|Vijayawada|Andhra Pradesh|2025-09-03|     Shipped|\n",
            "|  100365|      C0005| Sakshi Joshi|      Wireless Mouse|    Electronics|       0|       780|              15|             UPI|      Pune|   Maharashtra|2025-03-04|   Delivered|\n",
            "|  100375|      C0027|   Harsh Jain|    Basmati Rice 5kg|        Grocery|       0|       460|              10|      Debit Card|    Mumbai|   Maharashtra|2025-05-03|     Shipped|\n",
            "|  100458|      C0216|  Lakshmi Das|            Football|         Sports|      -2|       800|              12|      Debit Card|Vijayawada|Andhra Pradesh|2026-09-03|     Shipped|\n",
            "|  100501|      C0226|   Priya Bose|    Badminton Racket|         Sports|       0|       880|              35|Cash on Delivery|      Pune|   Maharashtra|2025-10-29|   Delivered|\n",
            "|  100593|      C0094|    Sai Patel|         Coffee 500g|        Grocery|      -1|       890|               5|      Debit Card|   Lucknow| Uttar Pradesh|2026-05-06|   Delivered|\n",
            "|  100780|      C0235| Varun Kapoor|        Office Chair|      Furniture|       0|     15610|               0|     Credit Card|   Lucknow| Uttar Pradesh|2025-02-22|   Delivered|\n",
            "|  100864|      C0147| Deepak Kumar|Competitive Exam ...|          Books|      -2|       970|              30|Cash on Delivery| Hyderabad|     Telangana|2025-12-07|   Delivered|\n",
            "|  100919|      C0214|Neha Tripathi|              Jacket|       Clothing|      -2|      3600|              20|             UPI|    Jaipur|     Rajasthan|2025-02-06|   Delivered|\n",
            "|  100926|      C0002|  Deepak Jain|         Ceiling Fan|Home Appliances|      -2|      3070|              40|     Net Banking|      Pune|   Maharashtra|2026-08-19|   Delivered|\n",
            "+--------+-----------+-------------+--------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+\n",
            "\n",
            "Invalid unit price:\n",
            "+--------+-----------+----------------+------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+\n",
            "|order_id|customer_id|   customer_name|     product|   category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|\n",
            "+--------+-----------+----------------+------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+\n",
            "|  100235|      C0349|     Tanvi Reddy|  Power Bank|Electronics|       7|      -500|               0|           UPI|Vijayawada|Andhra Pradesh|2026-03-02|   Delivered|\n",
            "|  100284|      C0302|    Ananya Reddy| Study Table|  Furniture|       2|      -500|              30|    Debit Card|Vijayawada|Andhra Pradesh|2026-06-22|   Delivered|\n",
            "|  100487|      C0232|Keerthi Kulkarni|Office Chair|  Furniture|       4|      -500|              25|    Debit Card|    Mumbai|   Maharashtra|2025-02-05|    Returned|\n",
            "|  100930|      C0056|   Sanjay Shetty|Formal Shirt|   Clothing|       8|      -250|              35|   Credit Card|Coimbatore|    Tamil Nadu|2026-07-28|   Delivered|\n",
            "+--------+-----------+----------------+------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+\n",
            "\n",
            "Invalid discount percentage:\n",
            "+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+\n",
            "|order_id|customer_id|  customer_name|         product|       category|quantity|unit_price|discount_percent|  payment_method|      city|     state|order_date|order_status|\n",
            "+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+\n",
            "|  100034|      C0192|   Divya Sharma|        Yoga Mat|         Sports|       7|      1720|              -5|Cash on Delivery|Coimbatore|Tamil Nadu|2025-08-19|   Delivered|\n",
            "|  100140|      C0075|Swathi Tripathi|  Cooking Oil 5L|        Grocery|       4|       790|              -5|     Credit Card| Hyderabad| Telangana|2025-10-01|     Shipped|\n",
            "|  100217|      C0214|  Neha Tripathi|        Football|         Sports|       1|       650|              -5|             UPI|   Chennai|Tamil Nadu|2026-06-14|   Delivered|\n",
            "|  100310|      C0332|  Meera Agarwal|  Vacuum Cleaner|Home Appliances|       9|      9280|             120|      Debit Card| Bengaluru| Karnataka|2025-04-12|   Cancelled|\n",
            "|  100363|      C0030|     Sneha Nair|Basmati Rice 5kg|        Grocery|       3|       900|             120|      Debit Card|     Delhi|     Delhi|2026-03-09|   Delivered|\n",
            "|  100574|      C0287|  Anjali Chopra|       Biography|          Books|       9|       860|             120|     Credit Card| Hyderabad| Telangana|2025-04-19|   Delivered|\n",
            "|  100857|      C0090| Abhinav Pillai|   Bedside Table|      Furniture|       4|      2460|             120|      Debit Card|Coimbatore|Tamil Nadu|2025-07-19|   Delivered|\n",
            "|  100999|      C0017| Siddharth Nair|      Hair Serum|         Beauty|       1|       940|              -5|Cash on Delivery|     Delhi|     Delhi|2026-08-15|     Shipped|\n",
            "+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+\n",
            "\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "from pyspark.sql.functions import col, trim, lower, count\n",
        "\n",
        "# Check text fields for whitespace/capitalization inconsistencies\n",
        "text_columns = [\n",
        "    \"customer_name\",\n",
        "    \"product\",\n",
        "    \"category\",\n",
        "    \"payment_method\",\n",
        "    \"city\",\n",
        "    \"state\",\n",
        "    \"order_status\"\n",
        "]\n",
        "\n",
        "for c in text_columns:\n",
        "    print(f\"\\n===== {c} =====\")\n",
        "    df.groupBy(col(c)).count().orderBy(col(c)).show(30, truncate=False)"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "7CBykp-PLkFj",
        "outputId": "50715f85-4fe7-4ace-e774-e9f6352675fd"
      },
      "execution_count": 11,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "\n",
            "===== customer_name =====\n",
            "+----------------+-----+\n",
            "|customer_name   |count|\n",
            "+----------------+-----+\n",
            "|NULL            |10   |\n",
            "|Aarav Agarwal   |2    |\n",
            "|Aarav Chopra    |3    |\n",
            "|Aarav Iyer      |2    |\n",
            "|Aarav Kulkarni  |5    |\n",
            "|Aarav Kumar     |2    |\n",
            "|Aarav Nair      |2    |\n",
            "|Aarav Patel     |6    |\n",
            "|Aarav Singh     |4    |\n",
            "|Aarav Tripathi  |2    |\n",
            "|Abhinav Gupta   |6    |\n",
            "|Abhinav Kulkarni|3    |\n",
            "|Abhinav Malhotra|4    |\n",
            "|Abhinav Pandey  |2    |\n",
            "|Abhinav Pillai  |6    |\n",
            "|Abhinav Rao     |6    |\n",
            "|Abhinav Yadav   |4    |\n",
            "|Aditi Agarwal   |1    |\n",
            "|Aditi Chopra    |6    |\n",
            "|Aditi Kapoor    |2    |\n",
            "|Aditi Mishra    |3    |\n",
            "|Aditi Patel     |9    |\n",
            "|Aditi Tripathi  |1    |\n",
            "|Aditya Chopra   |5    |\n",
            "|Aditya Malhotra |2    |\n",
            "|Aditya Mehta    |2    |\n",
            "|Aditya Naidu    |1    |\n",
            "|Aditya Rao      |3    |\n",
            "|Akash Jain      |1    |\n",
            "|Akash Jain      |1    |\n",
            "+----------------+-----+\n",
            "only showing top 30 rows\n",
            "\n",
            "===== product =====\n",
            "+----------------------+-----+\n",
            "|product               |count|\n",
            "+----------------------+-----+\n",
            "|Air Fryer             |16   |\n",
            "|Atta 10kg             |22   |\n",
            "|Badminton Racket      |15   |\n",
            "|Basmati Rice 5kg      |18   |\n",
            "|Bedside Table         |24   |\n",
            "|Biography             |20   |\n",
            "|Bluetooth Speaker     |6    |\n",
            "|Bookshelf             |19   |\n",
            "|Business Book         |19   |\n",
            "|Ceiling Fan           |21   |\n",
            "|Coffee 500g           |18   |\n",
            "|Competitive Exam Guide|30   |\n",
            "|Cooking Oil 5L        |19   |\n",
            "|Cricket Bat           |14   |\n",
            "|Data Science Handbook |12   |\n",
            "|Dining Table          |18   |\n",
            "|Dry Fruits Pack       |12   |\n",
            "|Dumbbell Set          |21   |\n",
            "|Electric Kettle       |17   |\n",
            "|Face Wash             |25   |\n",
            "|Football              |23   |\n",
            "|Formal Shirt          |12   |\n",
            "|Hair Serum            |16   |\n",
            "|Hoodie                |14   |\n",
            "|Induction Cooktop     |20   |\n",
            "|Jacket                |16   |\n",
            "|Jeans                 |13   |\n",
            "|Kurta                 |12   |\n",
            "|Laptop                |15   |\n",
            "|Makeup Kit            |30   |\n",
            "+----------------------+-----+\n",
            "only showing top 30 rows\n",
            "\n",
            "===== category =====\n",
            "+---------------+-----+\n",
            "|category       |count|\n",
            "+---------------+-----+\n",
            "|Beauty         |136  |\n",
            "|Books          |128  |\n",
            "|CLOTHING       |1    |\n",
            "|Clothing       |108  |\n",
            "|ELECTRONICS    |1    |\n",
            "|Electronics    |103  |\n",
            "|Furniture      |132  |\n",
            "|Grocery        |136  |\n",
            "|Home Appliances|115  |\n",
            "|SPORTS         |1    |\n",
            "|Sports         |136  |\n",
            "|beauty         |2    |\n",
            "|sports         |1    |\n",
            "+---------------+-----+\n",
            "\n",
            "\n",
            "===== payment_method =====\n",
            "+----------------+-----+\n",
            "|payment_method  |count|\n",
            "+----------------+-----+\n",
            "|NULL            |8    |\n",
            "|Cash on Delivery|208  |\n",
            "|Credit Card     |207  |\n",
            "|Debit Card      |201  |\n",
            "|Net Banking     |206  |\n",
            "|UPI             |170  |\n",
            "+----------------+-----+\n",
            "\n",
            "\n",
            "===== city =====\n",
            "+-----------+-----+\n",
            "|city       |count|\n",
            "+-----------+-----+\n",
            "|NULL       |8    |\n",
            "| Hyderabad |2    |\n",
            "| Mumbai    |1    |\n",
            "|Bengaluru  |91   |\n",
            "|Chennai    |88   |\n",
            "|Coimbatore |81   |\n",
            "|DELHI      |2    |\n",
            "|Delhi      |81   |\n",
            "|HYDERABAD  |1    |\n",
            "|Hyderabad  |94   |\n",
            "|JAIPUR     |1    |\n",
            "|Jaipur     |84   |\n",
            "|Kolkata    |81   |\n",
            "|Lucknow    |90   |\n",
            "|Mumbai     |106  |\n",
            "|Pune       |82   |\n",
            "|Vijayawada |105  |\n",
            "|coimbatore |1    |\n",
            "|delhi      |1    |\n",
            "+-----------+-----+\n",
            "\n",
            "\n",
            "===== state =====\n",
            "+--------------+-----+\n",
            "|state         |count|\n",
            "+--------------+-----+\n",
            "|Andhra Pradesh|106  |\n",
            "|Delhi         |85   |\n",
            "|Karnataka     |94   |\n",
            "|Maharashtra   |190  |\n",
            "|Rajasthan     |86   |\n",
            "|Tamil Nadu    |170  |\n",
            "|Telangana     |98   |\n",
            "|Uttar Pradesh |90   |\n",
            "|West Bengal   |81   |\n",
            "+--------------+-----+\n",
            "\n",
            "\n",
            "===== order_status =====\n",
            "+------------+-----+\n",
            "|order_status|count|\n",
            "+------------+-----+\n",
            "|Cancelled   |77   |\n",
            "|Delivered   |587  |\n",
            "|Processing  |111  |\n",
            "|Returned    |57   |\n",
            "|Shipped     |168  |\n",
            "+------------+-----+\n",
            "\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "from pyspark.sql.functions import col, try_to_timestamp, lit\n",
        "\n",
        "date_check = df.withColumn(\n",
        "    \"parsed_order_date\",\n",
        "    try_to_timestamp(col(\"order_date\"), lit(\"yyyy-MM-dd\"))\n",
        ")\n",
        "\n",
        "print(\"Invalid/unparseable dates:\")\n",
        "\n",
        "date_check.filter(\n",
        "    col(\"order_date\").isNotNull() &\n",
        "    col(\"parsed_order_date\").isNull()\n",
        ").select(\n",
        "    \"order_id\",\n",
        "    \"order_date\"\n",
        ").show(50, truncate=False)"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "nzMu0RP5L9DM",
        "outputId": "c624e82d-ec39-4ae3-e7c3-f1cd3b04dbb1"
      },
      "execution_count": 16,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Invalid/unparseable dates:\n",
            "+--------+----------+\n",
            "|order_id|order_date|\n",
            "+--------+----------+\n",
            "|100097  |31/02/2026|\n",
            "|100358  |2025/07/14|\n",
            "|100448  |not-a-date|\n",
            "|100634  |2025-02-30|\n",
            "|100637  |2026-13-05|\n",
            "|100969  |09-15-2026|\n",
            "+--------+----------+\n",
            "\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "from pyspark.sql.functions import (\n",
        "    col, trim, initcap, upper, regexp_replace,\n",
        "    when, coalesce, to_date, try_to_timestamp, lit\n",
        ")\n",
        "\n",
        "# Start from the original Bronze DataFrame\n",
        "df_clean = df.dropDuplicates()\n",
        "\n",
        "print(\"Rows after removing exact duplicates:\", df_clean.count())\n",
        "\n",
        "\n",
        "# ---------------------------------------------------------\n",
        "# 1. Clean text fields\n",
        "# ---------------------------------------------------------\n",
        "\n",
        "text_columns = [\n",
        "    \"customer_name\",\n",
        "    \"product\",\n",
        "    \"category\",\n",
        "    \"payment_method\",\n",
        "    \"city\",\n",
        "    \"state\",\n",
        "    \"order_status\"\n",
        "]\n",
        "\n",
        "for c in text_columns:\n",
        "    df_clean = df_clean.withColumn(\n",
        "        c,\n",
        "        initcap(trim(col(c)))\n",
        "    )\n",
        "\n",
        "\n",
        "# ---------------------------------------------------------\n",
        "# 2. Handle missing text values\n",
        "# ---------------------------------------------------------\n",
        "\n",
        "for c in text_columns:\n",
        "    df_clean = df_clean.withColumn(\n",
        "        c,\n",
        "        when(\n",
        "            col(c).isNull() | (trim(col(c)) == \"\"),\n",
        "            \"Unknown\"\n",
        "        ).otherwise(col(c))\n",
        "    )\n",
        "\n",
        "\n",
        "# ---------------------------------------------------------\n",
        "# 3. Validate quantity\n",
        "#    Quantity must be greater than 0\n",
        "# ---------------------------------------------------------\n",
        "\n",
        "df_clean = df_clean.withColumn(\n",
        "    \"quantity\",\n",
        "    when(col(\"quantity\") > 0, col(\"quantity\"))\n",
        "    .otherwise(None)\n",
        ")\n",
        "\n",
        "\n",
        "# ---------------------------------------------------------\n",
        "# 4. Validate unit price\n",
        "#    Unit price cannot be negative\n",
        "# ---------------------------------------------------------\n",
        "\n",
        "df_clean = df_clean.withColumn(\n",
        "    \"unit_price\",\n",
        "    when(col(\"unit_price\") >= 0, col(\"unit_price\"))\n",
        "    .otherwise(None)\n",
        ")\n",
        "\n",
        "\n",
        "# ---------------------------------------------------------\n",
        "# 5. Validate discount percentage\n",
        "#    Valid range: 0 to 100\n",
        "# ---------------------------------------------------------\n",
        "\n",
        "df_clean = df_clean.withColumn(\n",
        "    \"discount_percent\",\n",
        "    when(\n",
        "        (col(\"discount_percent\") >= 0) &\n",
        "        (col(\"discount_percent\") <= 100),\n",
        "        col(\"discount_percent\")\n",
        "    ).otherwise(None)\n",
        ")\n",
        "\n",
        "\n",
        "# ---------------------------------------------------------\n",
        "# 6. Handle missing numerical values\n",
        "# ---------------------------------------------------------\n",
        "\n",
        "# Use median values for numerical columns\n",
        "quantity_median = df_clean.approxQuantile(\n",
        "    \"quantity\", [0.5], 0.01\n",
        ")[0]\n",
        "\n",
        "unit_price_median = df_clean.approxQuantile(\n",
        "    \"unit_price\", [0.5], 0.01\n",
        ")[0]\n",
        "\n",
        "discount_median = df_clean.approxQuantile(\n",
        "    \"discount_percent\", [0.5], 0.01\n",
        ")[0]\n",
        "\n",
        "df_clean = df_clean.fillna({\n",
        "    \"quantity\": quantity_median,\n",
        "    \"unit_price\": unit_price_median,\n",
        "    \"discount_percent\": discount_median\n",
        "})\n",
        "\n",
        "\n",
        "# ---------------------------------------------------------\n",
        "# 7. Clean and standardize order_date\n",
        "# ---------------------------------------------------------\n",
        "\n",
        "# Try the expected format first.\n",
        "# Invalid dates become NULL instead of causing an error.\n",
        "\n",
        "df_clean = df_clean.withColumn(\n",
        "    \"parsed_date\",\n",
        "    try_to_timestamp(\n",
        "        col(\"order_date\"),\n",
        "        lit(\"yyyy-MM-dd\")\n",
        "    )\n",
        ")\n",
        "\n",
        "# Try a few common alternative formats\n",
        "df_clean = df_clean.withColumn(\n",
        "    \"parsed_date\",\n",
        "    coalesce(\n",
        "        col(\"parsed_date\"),\n",
        "        try_to_timestamp(col(\"order_date\"), lit(\"yyyy/MM/dd\")),\n",
        "        try_to_timestamp(col(\"order_date\"), lit(\"dd-MM-yyyy\")),\n",
        "        try_to_timestamp(col(\"order_date\"), lit(\"MM-dd-yyyy\")),\n",
        "        try_to_timestamp(col(\"order_date\"), lit(\"dd/MM/yyyy\"))\n",
        "    )\n",
        ")\n",
        "\n",
        "# Convert to consistent yyyy-MM-dd format\n",
        "df_clean = df_clean.withColumn(\n",
        "    \"order_date\",\n",
        "    when(\n",
        "        col(\"parsed_date\").isNotNull(),\n",
        "        col(\"parsed_date\").cast(\"date\")\n",
        "    )\n",
        ")\n",
        "\n",
        "# Remove temporary column\n",
        "df_clean = df_clean.drop(\"parsed_date\")\n",
        "\n",
        "\n",
        "# ---------------------------------------------------------\n",
        "# 8. Display result\n",
        "# ---------------------------------------------------------\n",
        "\n",
        "print(\"Final rows after preprocessing:\", df_clean.count())\n",
        "\n",
        "df_clean.show(10, truncate=False)"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "LRs338V1MDwD",
        "outputId": "6ede6cb1-0ee6-41fc-ca0c-3c17dd7ab88a"
      },
      "execution_count": 17,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Rows after removing exact duplicates: 990\n",
            "Final rows after preprocessing: 990\n",
            "+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+\n",
            "|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|\n",
            "+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+\n",
            "|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |\n",
            "|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |\n",
            "|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books          |6       |610       |12              |Upi             |Coimbatore|Tamil Nadu    |2025-06-02|Delivered   |\n",
            "|100931  |C0204      |Pranav Reddy      |Dining Table          |Furniture      |10      |37710     |30              |Credit Card     |Pune      |Maharashtra   |2025-12-25|Cancelled   |\n",
            "|100168  |C0179      |Sai Sharma        |Office Chair          |Furniture      |4       |6820      |35              |Cash On Delivery|Coimbatore|Tamil Nadu    |2026-06-04|Processing  |\n",
            "|100473  |C0195      |Tanvi Gupta       |Badminton Racket      |Sports         |2       |840       |35              |Debit Card      |Kolkata   |West Bengal   |2026-03-24|Shipped     |\n",
            "|100532  |C0149      |Manish Pillai     |Shampoo               |Beauty         |4       |770       |12              |Cash On Delivery|Delhi     |Delhi         |2026-04-30|Delivered   |\n",
            "|100011  |C0021      |Keerthi Tripathi  |Programming Book      |Books          |3       |1140      |35              |Net Banking     |Delhi     |Delhi         |2026-07-27|Delivered   |\n",
            "|100064  |C0276      |Ananya Agarwal    |Microwave Oven        |Home Appliances|3       |16610     |0               |Upi             |Hyderabad |Telangana     |2025-05-15|Delivered   |\n",
            "|100176  |C0138      |Pranav Joshi      |Electric Kettle       |Home Appliances|2       |2170      |35              |Net Banking     |Vijayawada|Andhra Pradesh|2026-08-23|Delivered   |\n",
            "+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+\n",
            "only showing top 10 rows\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "print(\"Invalid quantity:\")\n",
        "df_clean.filter(col(\"quantity\") <= 0).show()\n",
        "\n",
        "print(\"Invalid unit price:\")\n",
        "df_clean.filter(col(\"unit_price\") < 0).show()\n",
        "\n",
        "print(\"Invalid discount:\")\n",
        "df_clean.filter(\n",
        "    (col(\"discount_percent\") < 0) |\n",
        "    (col(\"discount_percent\") > 100)\n",
        ").show()"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "ctJQrhJ3MRID",
        "outputId": "905b61ab-1570-4700-d38f-b2d11c47027a"
      },
      "execution_count": 18,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Invalid quantity:\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "\n",
            "Invalid unit price:\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "\n",
            "Invalid discount:\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "print(\"Rows with missing/invalid order dates:\")\n",
        "\n",
        "df_clean.filter(\n",
        "    col(\"order_date\").isNull()\n",
        ").select(\n",
        "    \"order_id\",\n",
        "    \"order_date\"\n",
        ").show(50, truncate=False)"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "DIlmGBIsMZcn",
        "outputId": "4b016ec4-b206-4093-d43f-51e9894c220c"
      },
      "execution_count": 19,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Rows with missing/invalid order dates:\n",
            "+--------+----------+\n",
            "|order_id|order_date|\n",
            "+--------+----------+\n",
            "|100097  |NULL      |\n",
            "|100637  |NULL      |\n",
            "|100448  |NULL      |\n",
            "|100634  |NULL      |\n",
            "+--------+----------+\n",
            "\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "from pyspark.sql.functions import sum\n",
        "\n",
        "print(\"NULL values after preprocessing:\")\n",
        "\n",
        "df_clean.select([\n",
        "    sum(col(c).isNull().cast(\"int\")).alias(c)\n",
        "    for c in df_clean.columns\n",
        "]).show()"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "CfAQhfB2Mfjf",
        "outputId": "ed4aacfa-3580-4fd1-9884-b70764a8b6cf"
      },
      "execution_count": 20,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "NULL values after preprocessing:\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "|       0|          0|            0|      0|       0|       0|         0|               0|             0|   0|    0|         4|           0|\n",
            "+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+\n",
            "\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "print(\"Final row count:\", df_clean.count())\n",
        "\n",
        "print(\"\\nFinal schema:\")\n",
        "df_clean.printSchema()\n",
        "\n",
        "print(\"\\nFinal sample:\")\n",
        "df_clean.show(10, truncate=False)"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "T9UpbvlbMg-W",
        "outputId": "611f010e-aa3f-41b0-aa3d-0fc7ff52d745"
      },
      "execution_count": 21,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Final row count: 990\n",
            "\n",
            "Final schema:\n",
            "root\n",
            " |-- order_id: integer (nullable = true)\n",
            " |-- customer_id: string (nullable = true)\n",
            " |-- customer_name: string (nullable = true)\n",
            " |-- product: string (nullable = true)\n",
            " |-- category: string (nullable = true)\n",
            " |-- quantity: integer (nullable = true)\n",
            " |-- unit_price: integer (nullable = true)\n",
            " |-- discount_percent: integer (nullable = true)\n",
            " |-- payment_method: string (nullable = true)\n",
            " |-- city: string (nullable = true)\n",
            " |-- state: string (nullable = true)\n",
            " |-- order_date: date (nullable = true)\n",
            " |-- order_status: string (nullable = true)\n",
            "\n",
            "\n",
            "Final sample:\n",
            "+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+\n",
            "|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|\n",
            "+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+\n",
            "|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |\n",
            "|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |\n",
            "|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books          |6       |610       |12              |Upi             |Coimbatore|Tamil Nadu    |2025-06-02|Delivered   |\n",
            "|100931  |C0204      |Pranav Reddy      |Dining Table          |Furniture      |10      |37710     |30              |Credit Card     |Pune      |Maharashtra   |2025-12-25|Cancelled   |\n",
            "|100168  |C0179      |Sai Sharma        |Office Chair          |Furniture      |4       |6820      |35              |Cash On Delivery|Coimbatore|Tamil Nadu    |2026-06-04|Processing  |\n",
            "|100473  |C0195      |Tanvi Gupta       |Badminton Racket      |Sports         |2       |840       |35              |Debit Card      |Kolkata   |West Bengal   |2026-03-24|Shipped     |\n",
            "|100532  |C0149      |Manish Pillai     |Shampoo               |Beauty         |4       |770       |12              |Cash On Delivery|Delhi     |Delhi         |2026-04-30|Delivered   |\n",
            "|100011  |C0021      |Keerthi Tripathi  |Programming Book      |Books          |3       |1140      |35              |Net Banking     |Delhi     |Delhi         |2026-07-27|Delivered   |\n",
            "|100064  |C0276      |Ananya Agarwal    |Microwave Oven        |Home Appliances|3       |16610     |0               |Upi             |Hyderabad |Telangana     |2025-05-15|Delivered   |\n",
            "|100176  |C0138      |Pranav Joshi      |Electric Kettle       |Home Appliances|2       |2170      |35              |Net Banking     |Vijayawada|Andhra Pradesh|2026-08-23|Delivered   |\n",
            "+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+\n",
            "only showing top 10 rows\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "import os\n",
        "import shutil\n",
        "import glob\n",
        "\n",
        "# Temporary Spark output folder\n",
        "temp_output = \"silver_temp\"\n",
        "\n",
        "# Remove old output if it exists\n",
        "if os.path.exists(temp_output):\n",
        "    shutil.rmtree(temp_output)\n",
        "\n",
        "# Write the cleaned DataFrame as a single CSV\n",
        "df_clean.coalesce(1).write \\\n",
        "    .mode(\"overwrite\") \\\n",
        "    .option(\"header\", True) \\\n",
        "    .csv(temp_output)\n",
        "\n",
        "# Find the generated CSV file\n",
        "csv_file = glob.glob(f\"{temp_output}/part-*.csv\")[0]\n",
        "\n",
        "# Create the required silver folder\n",
        "os.makedirs(\"silver\", exist_ok=True)\n",
        "\n",
        "# Copy/rename Spark's part file to the required filename\n",
        "final_file = \"silver/sales_silver.csv\"\n",
        "shutil.copy(csv_file, final_file)\n",
        "\n",
        "print(\"Silver file created successfully!\")\n",
        "print(\"File:\", final_file)\n",
        "print(\"Size:\", os.path.getsize(final_file), \"bytes\")"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "Gzpx_wkfMui5",
        "outputId": "d8db363a-3d6d-480a-80ca-da9191ab4950"
      },
      "execution_count": 22,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Silver file created successfully!\n",
            "File: silver/sales_silver.csv\n",
            "Size: 107544 bytes\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "import pandas as pd\n",
        "\n",
        "silver_preview = pd.read_csv(\"silver/sales_silver.csv\")\n",
        "\n",
        "print(\"Rows:\", len(silver_preview))\n",
        "print(\"Columns:\", len(silver_preview.columns))\n",
        "\n",
        "silver_preview.head(10)"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 397
        },
        "id": "whrgkwqVMzN0",
        "outputId": "20349d7b-8b8e-4c49-af9d-968ff8a40c37"
      },
      "execution_count": 23,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Rows: 990\n",
            "Columns: 13\n"
          ]
        },
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "   order_id customer_id       customer_name                 product  \\\n",
              "0    100331       C0005        Sakshi Joshi                Football   \n",
              "1    100333       C0263  Siddharth Kulkarni        Programming Book   \n",
              "2    100506       C0228       Diya Tripathi  Competitive Exam Guide   \n",
              "3    100931       C0204        Pranav Reddy            Dining Table   \n",
              "4    100168       C0179          Sai Sharma            Office Chair   \n",
              "5    100473       C0195         Tanvi Gupta        Badminton Racket   \n",
              "6    100532       C0149       Manish Pillai                 Shampoo   \n",
              "7    100011       C0021    Keerthi Tripathi        Programming Book   \n",
              "8    100064       C0276      Ananya Agarwal          Microwave Oven   \n",
              "9    100176       C0138        Pranav Joshi         Electric Kettle   \n",
              "\n",
              "          category  quantity  unit_price  discount_percent    payment_method  \\\n",
              "0           Sports         2         670                30       Credit Card   \n",
              "1            Books         4         670                 0       Net Banking   \n",
              "2            Books         6         610                12               Upi   \n",
              "3        Furniture        10       37710                30       Credit Card   \n",
              "4        Furniture         4        6820                35  Cash On Delivery   \n",
              "5           Sports         2         840                35        Debit Card   \n",
              "6           Beauty         4         770                12  Cash On Delivery   \n",
              "7            Books         3        1140                35       Net Banking   \n",
              "8  Home Appliances         3       16610                 0               Upi   \n",
              "9  Home Appliances         2        2170                35       Net Banking   \n",
              "\n",
              "         city           state  order_date order_status  \n",
              "0   Bengaluru       Karnataka  2025-06-06    Cancelled  \n",
              "1        Pune     Maharashtra  2025-05-08    Delivered  \n",
              "2  Coimbatore      Tamil Nadu  2025-06-02    Delivered  \n",
              "3        Pune     Maharashtra  2025-12-25    Cancelled  \n",
              "4  Coimbatore      Tamil Nadu  2026-06-04   Processing  \n",
              "5     Kolkata     West Bengal  2026-03-24      Shipped  \n",
              "6       Delhi           Delhi  2026-04-30    Delivered  \n",
              "7       Delhi           Delhi  2026-07-27    Delivered  \n",
              "8   Hyderabad       Telangana  2025-05-15    Delivered  \n",
              "9  Vijayawada  Andhra Pradesh  2026-08-23    Delivered  "
            ],
            "text/html": [
              "\n",
              "  <div id=\"df-e4e4e441-c81c-46b4-9be5-b159c1cdd52f\" class=\"colab-df-container\">\n",
              "    <div>\n",
              "<style scoped>\n",
              "    .dataframe tbody tr th:only-of-type {\n",
              "        vertical-align: middle;\n",
              "    }\n",
              "\n",
              "    .dataframe tbody tr th {\n",
              "        vertical-align: top;\n",
              "    }\n",
              "\n",
              "    .dataframe thead th {\n",
              "        text-align: right;\n",
              "    }\n",
              "</style>\n",
              "<table border=\"1\" class=\"dataframe\">\n",
              "  <thead>\n",
              "    <tr style=\"text-align: right;\">\n",
              "      <th></th>\n",
              "      <th>order_id</th>\n",
              "      <th>customer_id</th>\n",
              "      <th>customer_name</th>\n",
              "      <th>product</th>\n",
              "      <th>category</th>\n",
              "      <th>quantity</th>\n",
              "      <th>unit_price</th>\n",
              "      <th>discount_percent</th>\n",
              "      <th>payment_method</th>\n",
              "      <th>city</th>\n",
              "      <th>state</th>\n",
              "      <th>order_date</th>\n",
              "      <th>order_status</th>\n",
              "    </tr>\n",
              "  </thead>\n",
              "  <tbody>\n",
              "    <tr>\n",
              "      <th>0</th>\n",
              "      <td>100331</td>\n",
              "      <td>C0005</td>\n",
              "      <td>Sakshi Joshi</td>\n",
              "      <td>Football</td>\n",
              "      <td>Sports</td>\n",
              "      <td>2</td>\n",
              "      <td>670</td>\n",
              "      <td>30</td>\n",
              "      <td>Credit Card</td>\n",
              "      <td>Bengaluru</td>\n",
              "      <td>Karnataka</td>\n",
              "      <td>2025-06-06</td>\n",
              "      <td>Cancelled</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>1</th>\n",
              "      <td>100333</td>\n",
              "      <td>C0263</td>\n",
              "      <td>Siddharth Kulkarni</td>\n",
              "      <td>Programming Book</td>\n",
              "      <td>Books</td>\n",
              "      <td>4</td>\n",
              "      <td>670</td>\n",
              "      <td>0</td>\n",
              "      <td>Net Banking</td>\n",
              "      <td>Pune</td>\n",
              "      <td>Maharashtra</td>\n",
              "      <td>2025-05-08</td>\n",
              "      <td>Delivered</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>2</th>\n",
              "      <td>100506</td>\n",
              "      <td>C0228</td>\n",
              "      <td>Diya Tripathi</td>\n",
              "      <td>Competitive Exam Guide</td>\n",
              "      <td>Books</td>\n",
              "      <td>6</td>\n",
              "      <td>610</td>\n",
              "      <td>12</td>\n",
              "      <td>Upi</td>\n",
              "      <td>Coimbatore</td>\n",
              "      <td>Tamil Nadu</td>\n",
              "      <td>2025-06-02</td>\n",
              "      <td>Delivered</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>3</th>\n",
              "      <td>100931</td>\n",
              "      <td>C0204</td>\n",
              "      <td>Pranav Reddy</td>\n",
              "      <td>Dining Table</td>\n",
              "      <td>Furniture</td>\n",
              "      <td>10</td>\n",
              "      <td>37710</td>\n",
              "      <td>30</td>\n",
              "      <td>Credit Card</td>\n",
              "      <td>Pune</td>\n",
              "      <td>Maharashtra</td>\n",
              "      <td>2025-12-25</td>\n",
              "      <td>Cancelled</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>4</th>\n",
              "      <td>100168</td>\n",
              "      <td>C0179</td>\n",
              "      <td>Sai Sharma</td>\n",
              "      <td>Office Chair</td>\n",
              "      <td>Furniture</td>\n",
              "      <td>4</td>\n",
              "      <td>6820</td>\n",
              "      <td>35</td>\n",
              "      <td>Cash On Delivery</td>\n",
              "      <td>Coimbatore</td>\n",
              "      <td>Tamil Nadu</td>\n",
              "      <td>2026-06-04</td>\n",
              "      <td>Processing</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>5</th>\n",
              "      <td>100473</td>\n",
              "      <td>C0195</td>\n",
              "      <td>Tanvi Gupta</td>\n",
              "      <td>Badminton Racket</td>\n",
              "      <td>Sports</td>\n",
              "      <td>2</td>\n",
              "      <td>840</td>\n",
              "      <td>35</td>\n",
              "      <td>Debit Card</td>\n",
              "      <td>Kolkata</td>\n",
              "      <td>West Bengal</td>\n",
              "      <td>2026-03-24</td>\n",
              "      <td>Shipped</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>6</th>\n",
              "      <td>100532</td>\n",
              "      <td>C0149</td>\n",
              "      <td>Manish Pillai</td>\n",
              "      <td>Shampoo</td>\n",
              "      <td>Beauty</td>\n",
              "      <td>4</td>\n",
              "      <td>770</td>\n",
              "      <td>12</td>\n",
              "      <td>Cash On Delivery</td>\n",
              "      <td>Delhi</td>\n",
              "      <td>Delhi</td>\n",
              "      <td>2026-04-30</td>\n",
              "      <td>Delivered</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>7</th>\n",
              "      <td>100011</td>\n",
              "      <td>C0021</td>\n",
              "      <td>Keerthi Tripathi</td>\n",
              "      <td>Programming Book</td>\n",
              "      <td>Books</td>\n",
              "      <td>3</td>\n",
              "      <td>1140</td>\n",
              "      <td>35</td>\n",
              "      <td>Net Banking</td>\n",
              "      <td>Delhi</td>\n",
              "      <td>Delhi</td>\n",
              "      <td>2026-07-27</td>\n",
              "      <td>Delivered</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>8</th>\n",
              "      <td>100064</td>\n",
              "      <td>C0276</td>\n",
              "      <td>Ananya Agarwal</td>\n",
              "      <td>Microwave Oven</td>\n",
              "      <td>Home Appliances</td>\n",
              "      <td>3</td>\n",
              "      <td>16610</td>\n",
              "      <td>0</td>\n",
              "      <td>Upi</td>\n",
              "      <td>Hyderabad</td>\n",
              "      <td>Telangana</td>\n",
              "      <td>2025-05-15</td>\n",
              "      <td>Delivered</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>9</th>\n",
              "      <td>100176</td>\n",
              "      <td>C0138</td>\n",
              "      <td>Pranav Joshi</td>\n",
              "      <td>Electric Kettle</td>\n",
              "      <td>Home Appliances</td>\n",
              "      <td>2</td>\n",
              "      <td>2170</td>\n",
              "      <td>35</td>\n",
              "      <td>Net Banking</td>\n",
              "      <td>Vijayawada</td>\n",
              "      <td>Andhra Pradesh</td>\n",
              "      <td>2026-08-23</td>\n",
              "      <td>Delivered</td>\n",
              "    </tr>\n",
              "  </tbody>\n",
              "</table>\n",
              "</div>\n",
              "    <div class=\"colab-df-buttons\">\n",
              "\n",
              "  <div class=\"colab-df-container\">\n",
              "    <button class=\"colab-df-convert\" onclick=\"convertToInteractive('df-e4e4e441-c81c-46b4-9be5-b159c1cdd52f')\"\n",
              "            title=\"Convert this dataframe to an interactive table.\"\n",
              "            style=\"display:none;\">\n",
              "\n",
              "  <svg xmlns=\"http://www.w3.org/2000/svg\" height=\"24px\" viewBox=\"0 -960 960 960\">\n",
              "    <path d=\"M120-120v-720h720v720H120Zm60-500h600v-160H180v160Zm220 220h160v-160H400v160Zm0 220h160v-160H400v160ZM180-400h160v-160H180v160Zm440 0h160v-160H620v160ZM180-180h160v-160H180v160Zm440 0h160v-160H620v160Z\"/>\n",
              "  </svg>\n",
              "    </button>\n",
              "\n",
              "  <style>\n",
              "    .colab-df-container {\n",
              "      display:flex;\n",
              "      gap: 12px;\n",
              "    }\n",
              "\n",
              "    .colab-df-convert {\n",
              "      background-color: #E8F0FE;\n",
              "      border: none;\n",
              "      border-radius: 50%;\n",
              "      cursor: pointer;\n",
              "      display: none;\n",
              "      fill: #1967D2;\n",
              "      height: 32px;\n",
              "      padding: 0 0 0 0;\n",
              "      width: 32px;\n",
              "    }\n",
              "\n",
              "    .colab-df-convert:hover {\n",
              "      background-color: #E2EBFA;\n",
              "      box-shadow: 0px 1px 2px rgba(60, 64, 67, 0.3), 0px 1px 3px 1px rgba(60, 64, 67, 0.15);\n",
              "      fill: #174EA6;\n",
              "    }\n",
              "\n",
              "    .colab-df-buttons div {\n",
              "      margin-bottom: 4px;\n",
              "    }\n",
              "\n",
              "    [theme=dark] .colab-df-convert {\n",
              "      background-color: #3B4455;\n",
              "      fill: #D2E3FC;\n",
              "    }\n",
              "\n",
              "    [theme=dark] .colab-df-convert:hover {\n",
              "      background-color: #434B5C;\n",
              "      box-shadow: 0px 1px 3px 1px rgba(0, 0, 0, 0.15);\n",
              "      filter: drop-shadow(0px 1px 2px rgba(0, 0, 0, 0.3));\n",
              "      fill: #FFFFFF;\n",
              "    }\n",
              "  </style>\n",
              "\n",
              "    <script>\n",
              "      const buttonEl =\n",
              "        document.querySelector('#df-e4e4e441-c81c-46b4-9be5-b159c1cdd52f button.colab-df-convert');\n",
              "      buttonEl.style.display =\n",
              "        google.colab.kernel.accessAllowed ? 'block' : 'none';\n",
              "\n",
              "      async function convertToInteractive(key) {\n",
              "        const element = document.querySelector('#df-e4e4e441-c81c-46b4-9be5-b159c1cdd52f');\n",
              "        const dataTable =\n",
              "          await google.colab.kernel.invokeFunction('convertToInteractive',\n",
              "                                                    [key], {});\n",
              "        if (!dataTable) return;\n",
              "\n",
              "        const docLinkHtml = 'Like what you see? Visit the ' +\n",
              "          '<a target=\"_blank\" href=https://colab.research.google.com/notebooks/data_table.ipynb>data table notebook</a>'\n",
              "          + ' to learn more about interactive tables.';\n",
              "        element.innerHTML = '';\n",
              "        dataTable['output_type'] = 'display_data';\n",
              "        await google.colab.output.renderOutput(dataTable, element);\n",
              "        const docLink = document.createElement('div');\n",
              "        docLink.innerHTML = docLinkHtml;\n",
              "        element.appendChild(docLink);\n",
              "      }\n",
              "    </script>\n",
              "  </div>\n",
              "\n",
              "\n",
              "    </div>\n",
              "  </div>\n"
            ],
            "application/vnd.google.colaboratory.intrinsic+json": {
              "type": "dataframe",
              "variable_name": "silver_preview",
              "summary": "{\n  \"name\": \"silver_preview\",\n  \"rows\": 990,\n  \"fields\": [\n    {\n      \"column\": \"order_id\",\n      \"properties\": {\n        \"dtype\": \"number\",\n        \"std\": 287,\n        \"min\": 100001,\n        \"max\": 101000,\n        \"num_unique_values\": 982,\n        \"samples\": [\n          100820,\n          100383,\n          100039\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    },\n    {\n      \"column\": \"customer_id\",\n      \"properties\": {\n        \"dtype\": \"category\",\n        \"num_unique_values\": 335,\n        \"samples\": [\n          \"C0098\",\n          \"C0349\",\n          \"C0275\"\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    },\n    {\n      \"column\": \"customer_name\",\n      \"properties\": {\n        \"dtype\": \"category\",\n        \"num_unique_values\": 303,\n        \"samples\": [\n          \"Akash Saxena\",\n          \"Pooja Nair\",\n          \"Arjun Nair\"\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    },\n    {\n      \"column\": \"product\",\n      \"properties\": {\n        \"dtype\": \"category\",\n        \"num_unique_values\": 57,\n        \"samples\": [\n          \"Football\",\n          \"Badminton Racket\",\n          \"Cooking Oil 5l\"\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    },\n    {\n      \"column\": \"category\",\n      \"properties\": {\n        \"dtype\": \"category\",\n        \"num_unique_values\": 8,\n        \"samples\": [\n          \"Books\",\n          \"Grocery\",\n          \"Sports\"\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    },\n    {\n      \"column\": \"quantity\",\n      \"properties\": {\n        \"dtype\": \"number\",\n        \"std\": 2,\n        \"min\": 1,\n        \"max\": 10,\n        \"num_unique_values\": 10,\n        \"samples\": [\n          5,\n          4,\n          7\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    },\n    {\n      \"column\": \"unit_price\",\n      \"properties\": {\n        \"dtype\": \"number\",\n        \"std\": 11031,\n        \"min\": 0,\n        \"max\": 81010,\n        \"num_unique_values\": 549,\n        \"samples\": [\n          5800,\n          1400,\n          3870\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    },\n    {\n      \"column\": \"discount_percent\",\n      \"properties\": {\n        \"dtype\": \"number\",\n        \"std\": 12,\n        \"min\": 0,\n        \"max\": 40,\n        \"num_unique_values\": 10,\n        \"samples\": [\n          15,\n          0,\n          5\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    },\n    {\n      \"column\": \"payment_method\",\n      \"properties\": {\n        \"dtype\": \"category\",\n        \"num_unique_values\": 6,\n        \"samples\": [\n          \"Credit Card\",\n          \"Net Banking\",\n          \"Unknown\"\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    },\n    {\n      \"column\": \"city\",\n      \"properties\": {\n        \"dtype\": \"category\",\n        \"num_unique_values\": 12,\n        \"samples\": [\n          \"Lucknow\",\n          \"Jaipur\",\n          \"Bengaluru\"\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    },\n    {\n      \"column\": \"state\",\n      \"properties\": {\n        \"dtype\": \"category\",\n        \"num_unique_values\": 9,\n        \"samples\": [\n          \"Rajasthan\",\n          \"Maharashtra\",\n          \"Telangana\"\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    },\n    {\n      \"column\": \"order_date\",\n      \"properties\": {\n        \"dtype\": \"object\",\n        \"num_unique_values\": 522,\n        \"samples\": [\n          \"2026-09-21\",\n          \"2025-07-29\",\n          \"2026-04-30\"\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    },\n    {\n      \"column\": \"order_status\",\n      \"properties\": {\n        \"dtype\": \"category\",\n        \"num_unique_values\": 5,\n        \"samples\": [\n          \"Delivered\",\n          \"Returned\",\n          \"Processing\"\n        ],\n        \"semantic_type\": \"\",\n        \"description\": \"\"\n      }\n    }\n  ]\n}"
            }
          },
          "metadata": {},
          "execution_count": 23
        }
      ]
    }
  ]
}